# MindfulTech – Exploratory Analysis & ML Pipeline

**ML-Based Digital Habit & Wellbeing Platform**

This notebook documents the complete machine learning workflow for the MindfulTech project:
- Dataset loading & overview
- Exploratory Data Analysis (EDA)
- Feature Engineering
- Model Training & Evaluation
- Clustering & Anomaly Detection

> **Important:** The dataset used is **synthetic** and generated for educational purposes. It must not be presented as real-world survey data.

## 1. Import Libraries

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)
import joblib

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100
print('Libraries loaded successfully.')

## 2. Load Dataset

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'dataset.csv')
df = pd.read_csv(DATA_PATH)
print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(10)

## 3. Dataset Overview

In [ ]:
print('Shape:', df.shape)
print('\nColumn dtypes:')
print(df.dtypes)
print('\nFirst 5 rows:')
df.head()

## 4. Missing Values

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found.')

## 5. Descriptive Statistics

In [ ]:
df.describe().round(2)

## 6. Exploratory Data Analysis (EDA)

### 6.1 Distribution of Primary Features

In [ ]:
primary_features = ['age', 'screen_time', 'social_media', 'gaming', 'short_video',
                    'phone_unlocks', 'notifications', 'night_usage', 'sleep',
                    'focus_time', 'exercise', 'stress', 'productivity']

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(primary_features):
    sns.histplot(df[col], ax=axes[i], kde=True, color='teal', bins=25)
    axes[i].set_title(col, fontsize=10)
for j in range(len(primary_features), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Distribution of Primary Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 6.2 Box Plots by Risk Level

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(primary_features):
    sns.boxplot(data=df, x='risk_level', y=col, ax=axes[i],
                order=['Low', 'Moderate', 'High'], palette='RdYlGn_r')
    axes[i].set_title(col, fontsize=10)
for j in range(len(primary_features), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Feature Distributions by Risk Level', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. Visualization

### 7.1 Correlation Heatmap

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Feature Engineering

We compute 6 derived features from the 13 primary features:

| Derived Feature | Formula | Rationale |
|---|---|---|
| entertainment_hours | gaming + short_video | Total recreational entertainment |
| total_recreational_screen_time | social_media + gaming + short_video | Broader recreational usage |
| night_usage_ratio | night_usage / screen_time | Proportion of late-night usage |
| focus_to_screen_ratio | focus_time / screen_time | Productive vs total screen use |
| sleep_deficit_indicator | max(0, 7.5 - sleep) | Sleep debt relative to 7.5h |
| usage_intensity | phone_unlocks + notifications/10 | Distraction composite |

In [ ]:
# Verify derived features exist in the dataset
derived = ['entertainment_hours', 'total_recreational_screen_time',
           'night_usage_ratio', 'focus_to_screen_ratio',
           'sleep_deficit_indicator', 'usage_intensity']

for col in derived:
    if col in df.columns:
        print(f'  {col}: min={df[col].min():.2f}, max={df[col].max():.2f}, mean={df[col].mean():.2f}')
    else:
        print(f'  {col}: NOT FOUND in dataset')

print(f'\nTotal features available: {len(df.columns)}')

## 9. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Risk level distribution
risk_counts = df['risk_level'].value_counts()
colors = {'Low': '#22c55e', 'Moderate': '#f59e0b', 'High': '#ef4444'}
risk_counts.plot(kind='bar', ax=axes[0],
                 color=[colors.get(x, 'gray') for x in risk_counts.index])
axes[0].set_title('Risk Level Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(risk_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Wellbeing score distribution
if 'wellbeing_score' in df.columns:
    sns.histplot(df['wellbeing_score'], ax=axes[1], kde=True, color='teal', bins=30)
    axes[1].set_title('Wellbeing Score Distribution')

plt.tight_layout()
plt.show()

print('Risk level class balance:')
print(df['risk_level'].value_counts())
print(f'\nClass proportions:')
print((df['risk_level'].value_counts(normalize=True) * 100).round(1).astype(str) + '%')

## 10. Train/Test Split

In [ ]:
FEATURE_COLS = primary_features + derived
TARGET_COL = 'risk_level'

X = df[FEATURE_COLS].values
y_raw = df[TARGET_COL].values

le = LabelEncoder()
y = le.fit_transform(y_raw)
print('Label mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain set: {X_train.shape[0]} samples')
print(f'Test set:  {X_test.shape[0]} samples')

## 11. Preprocessing (Feature Scaling)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Feature scaling applied (StandardScaler).')
print(f'Train mean (first 3 features): {X_train_scaled[:, :3].mean(axis=0).round(4)}')
print(f'Train std  (first 3 features): {X_train_scaled[:, :3].std(axis=0).round(4)}')

## 12–15. Model Training & Evaluation

We train and evaluate four classifiers:
1. Logistic Regression (baseline)
2. Decision Tree (interpretable)
3. Random Forest (ensemble)
4. SVM (margin-based)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    print(f'\n{"="*50}')
    print(f'Training: {name}')
    print('='*50)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    
    results[name] = {'accuracy': acc, 'precision': prec, 'recall': rec,
                     'f1_score': f1, 'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std()}
    
    print(f'  Accuracy:  {acc:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}')
    print(f'  F1-Score:  {f1:.4f}')
    print(f'  CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    print(f'\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=le.classes_))

## 16. Model Comparison

In [ ]:
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.round(4)
print('\nModel Comparison Table:')
comparison_df

In [ ]:
# Model comparison bar chart
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1_score']
x = np.arange(len(results))
width = 0.18

fig, ax = plt.subplots(figsize=(10, 5))
for i, metric in enumerate(metrics_to_plot):
    values = [results[m][metric] for m in results]
    ax.bar(x + i * width, values, width, label=metric.replace('_', ' ').title())

ax.set_ylabel('Score')
ax.set_title('Model Comparison')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results.keys(), rotation=15, ha='right')
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

## 17. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
    disp.plot(ax=axes[i], cmap='Blues', colorbar=False)
    axes[i].set_title(f'{name}', fontsize=11)

plt.suptitle('Confusion Matrices', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 18. K-Means Clustering

K-Means identifies behavioural groups (not risk categories). We use k=4 clusters and assign labels based on centroid characteristics.

In [ ]:
# Elbow method
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_train_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, 'bo-', linewidth=2)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Train K-Means with k=4
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
km_final.fit(X_train_scaled)

# Analyse centroids
centroids = scaler.inverse_transform(km_final.cluster_centers_)
centroid_df = pd.DataFrame(centroids, columns=FEATURE_COLS)

print('Cluster Centroid Profiles (unscaled):')
centroid_df[primary_features].round(1)

In [ ]:
# Cluster size distribution
labels = km_final.predict(X_train_scaled)
unique, counts = np.unique(labels, return_counts=True)
print('Cluster sizes:')
for u, c in zip(unique, counts):
    print(f'  Cluster {u}: {c} samples ({c/len(labels)*100:.1f}%)')

# Visualise clusters (PCA 2D projection)
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis',
                      alpha=0.5, s=20)
plt.colorbar(scatter, label='Cluster')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('K-Means Clusters (PCA 2D Projection)')
plt.tight_layout()
plt.show()

## 19. Isolation Forest (Anomaly Detection)

Isolation Forest identifies statistically unusual usage patterns by measuring how easily an observation can be isolated from the rest of the dataset.

In [ ]:
iso = IsolationForest(n_estimators=100, contamination=0.08, random_state=42)
iso.fit(X_train_scaled)

# Predict anomalies on test set
anomaly_pred = iso.predict(X_test_scaled)
anomaly_scores = iso.decision_function(X_test_scaled)

n_normal = (anomaly_pred == 1).sum()
n_anomaly = (anomaly_pred == -1).sum()
print(f'Test set anomaly detection results:')
print(f'  Normal:  {n_normal} ({n_normal/len(anomaly_pred)*100:.1f}%)')
print(f'  Anomaly: {n_anomaly} ({n_anomaly/len(anomaly_pred)*100:.1f}%)')

# Visualise anomaly scores
plt.figure(figsize=(10, 4))
plt.hist(anomaly_scores, bins=40, color='teal', alpha=0.7, edgecolor='white')
plt.axvline(x=0, color='red', linestyle='--', label='Decision boundary')
plt.xlabel('Anomaly Score')
plt.ylabel('Count')
plt.title('Isolation Forest Anomaly Score Distribution')
plt.legend()
plt.tight_layout()
plt.show()

## 20. Final Model Selection

In [ ]:
# Determine best model by F1-score
best_name = max(results, key=lambda k: results[k]['f1_score'])
best_metrics = results[best_name]

print(f'Best performing model: {best_name}')
print(f'  Accuracy:     {best_metrics["accuracy"]:.4f}')
print(f'  F1-Score:     {best_metrics["f1_score"]:.4f}')
print(f'  CV Accuracy:  {best_metrics["cv_mean"]:.4f} ± {best_metrics["cv_std"]:.4f}')

print(f'\nNote: For the web application, Random Forest is used as the default')
print(f'classifier. The best model could be swapped in the prediction module.')

## 21. Save Model

Models are saved during the training pipeline (`ml/train_models.py`). Here we verify the saved files exist.

In [ ]:
MODEL_DIR = os.path.join('..', 'models')
expected_files = [
    'logistic_model.pkl', 'decision_tree_model.pkl',
    'random_forest_model.pkl', 'svm_model.pkl',
    'kmeans_model.pkl', 'isolation_forest.pkl',
    'scaler.pkl', 'label_encoder.pkl',
    'model_metrics.json', 'cluster_descriptions.json',
]

for f in expected_files:
    path = os.path.join(MODEL_DIR, f)
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    status = f'✓ {size:,} bytes' if exists else '✗ MISSING'
    print(f'  {f:35s} {status}')

## 22. Conclusion

### Summary

This notebook demonstrated the complete ML pipeline for **MindfulTech**:

1. **Dataset**: A synthetic dataset of 2,000 records with 13 primary features and 6 derived features was used. Risk labels were generated using weighted behavioural thresholds with added Gaussian noise to prevent perfect memorisation.

2. **Classification**: Four algorithms were trained and evaluated:
   - Logistic Regression (baseline)
   - Decision Tree (interpretable rules)
   - Random Forest (ensemble, used as default)
   - SVM (kernel-based)

3. **Evaluation**: All models were assessed using accuracy, precision, recall, F1-score, confusion matrices, and 5-fold stratified cross-validation.

4. **Clustering**: K-Means (k=4) identified behavioural groups with dynamically assigned labels based on centroid analysis.

5. **Anomaly Detection**: Isolation Forest flagged statistically unusual digital usage patterns.

### Limitations

- The dataset is **synthetic** — real-world data would likely produce different patterns and model performance.
- Model accuracy is modest due to intentional noise in the labelling process, which is realistic for self-reported behavioural data.
- This system provides **behavioural indicators**, not medical diagnoses.

### Disclaimer

> MindfulTech is an educational project. It does not diagnose addiction, depression, anxiety, ADHD, or any mental health condition. Recommendations are general wellbeing suggestions, not professional medical advice.